# LeetCode 2578 — Split With Minimum Sum bằng Python

Link đề: https://leetcode.com/problems/split-with-minimum-sum/ · Độ khó: **Easy** · Chủ đề: Math, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def splitNum(self, num: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tách các **chữ số** của `num` thành hai số `num1`, `num2` (được có chữ số `0` đứng đầu) sao cho **tổng `num1 + num2` nhỏ nhất**.
2. Biết biến số nguyên thành danh sách chữ số (`str(num)`, hoặc `% 10` / `// 10`) rồi ghép lại thành số (`int("07") = 7`).
3. Thấy ý tưởng **tham lam (greedy)**: chữ số **nhỏ** nên đứng ở hàng **lớn** (hàng chục, trăm...), và hai số nên có **độ dài gần bằng nhau**.
4. Viết 4 cách: vét cạn mọi hoán vị, sắp xếp rồi chia luân phiên bằng chuỗi, chia luân phiên bằng số, và **đếm chữ số** (counting) không cần sort.
5. Chứng minh ngắn gọn vì sao "sắp tăng dần rồi chia luân phiên" tối ưu (bất đẳng thức hoán vị).
6. Tránh các lỗi quen thuộc: không sắp xếp, sắp giảm dần, chia đôi dãy thay vì luân phiên, cộng chuỗi thay vì cộng số, dùng `set` làm mất chữ số lặp.

## 2. Đề bài & ví dụ

Cho số nguyên dương `num`. Hãy tách `num` thành hai số nguyên không âm `num1` và `num2` sao cho:

- Việc **nối** `num1` và `num2` là một **hoán vị** của `num` (tức tổng số lần xuất hiện của mỗi chữ số trong `num1` và `num2` bằng số lần chữ số đó xuất hiện trong `num`).
- `num1` và `num2` **được phép có chữ số 0 ở đầu**.

Trả về **tổng nhỏ nhất** có thể của `num1 + num2`.

Lưu ý: `num` đảm bảo không có chữ số 0 ở đầu; thứ tự các chữ số trong `num1`, `num2` có thể khác thứ tự trong `num`.

| Ví dụ | num | Output | Giải thích |
|---|---|---|---|
| 1 | `4325` | `59` | tách `num1 = 24`, `num2 = 35`, tổng `24 + 35 = 59` |
| 2 | `687` | `75` | tách `num1 = 68`, `num2 = 7`, tổng `68 + 7 = 75` |

Constraints: `10 <= num <= 10^9`.

(Hai ví dụ trên đã được đối chiếu lại bằng code vét cạn ở mục 5.)

In [1]:
num = 4325
print("các chữ số:", list(str(num)))
print("tách thử 24 + 35 =", 24 + 35)
print("tách thử 43 + 25 =", 43 + 25, "(lớn hơn)")
print("tách thử 25 + 34 =", 25 + 34)

các chữ số: ['4', '3', '2', '5']
tách thử 24 + 35 = 59
tách thử 43 + 25 = 68 (lớn hơn)
tách thử 25 + 34 = 59


## 3. Kiến thức nền

### 3.1. Từ số sang danh sách chữ số và ngược lại

`str(num)` đổi số thành chuỗi; duyệt chuỗi cho từng chữ số dạng ký tự; `int(ch)` đổi ký tự thành số. `int("07")` bằng `7` — chữ số `0` đứng đầu **không** gây lỗi, hợp với đề cho phép leading zero.

In [2]:
num = 4325
s = str(num)
print(s, list(s), [int(ch) for ch in s])
print(int("07"), int("00"), int("0123"))

4325 ['4', '3', '2', '5'] [4, 3, 2, 5]
7 0 123


### 3.2. `sorted` cho chuỗi / list

`sorted("4325")` cho list ký tự đã sắp theo thứ tự chữ cái: với chữ số `'0'..'9'` thứ tự này trùng thứ tự số. Muốn sắp giảm dần thêm `reverse=True`.

In [3]:
print(sorted("4325"))                    # ['2', '3', '4', '5']
print(sorted("4325", reverse=True))      # ['5', '4', '3', '2']
print(sorted("100200"))                  # ['0', '0', '0', '0', '1', '2']

['2', '3', '4', '5']
['5', '4', '3', '2']
['0', '0', '0', '0', '1', '2']


### 3.3. Slicing có bước: `digits[0::2]` và `digits[1::2]`

`x[a::2]` bắt đầu từ chỉ số `a`, nhảy mỗi lần 2 vị trí. `digits[0::2]` lấy các vị trí **chẵn** (0, 2, 4...), `digits[1::2]` lấy các vị trí **lẻ** (1, 3, 5...). Đây là cách **chia luân phiên** một dãy thành hai nửa.

In [4]:
digits = ['2', '3', '4', '5']
print(digits[0::2], digits[1::2])        # ['2', '4'] ['3', '5']
digits = ['6', '7', '8']
print(digits[0::2], digits[1::2])        # ['6', '8'] ['7']
print("".join(['6', '8']), int("".join(['6', '8'])))

['2', '4'] ['3', '5']
['6', '8'] ['7']
68 68


### 3.4. `itertools.permutations`

`permutations(seq)` sinh mọi cách **xếp thứ tự** các phần tử (kể cả phần tử bằng nhau cũng được coi như khác nhau). Với 4 chữ số có `4! = 24` hoán vị. Ta dùng nó cho lời giải vét cạn.

In [5]:
from itertools import permutations
perms = list(permutations("123"))
print(len(perms), perms)

6 [('1', '2', '3'), ('1', '3', '2'), ('2', '1', '3'), ('2', '3', '1'), ('3', '1', '2'), ('3', '2', '1')]


### 3.5. Lấy từng chữ số bằng `% 10`, `// 10` và mảng đếm

`x % 10` là chữ số cuối, `x // 10` bỏ chữ số cuối. Lặp tới khi `x` thành `0` ta duyệt mọi chữ số mà **không cần chuỗi**. Một mảng `cnt` dài 10 đếm số lần mỗi chữ số xuất hiện.

In [6]:
x = 4325
cnt = [0] * 10
while x:
    cnt[x % 10] += 1
    x //= 10
print(cnt)         # chữ số 2,3,4,5 mỗi chữ số 1 lần

[0, 0, 1, 1, 1, 1, 0, 0, 0, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tổng `num1 + num2` là tổng các (chữ số × giá trị hàng của nó). Chữ số đứng ở hàng **chục / trăm** đóng góp nhiều hơn rất nhiều so với ở hàng **đơn vị**. Vậy:

- Muốn tổng **nhỏ**, hãy để các chữ số **bé** ở các hàng **cao** và chữ số **lớn** ở các hàng **thấp**.
- Hai số nên có **độ dài gần bằng nhau**: nếu một số dài thêm một chữ số thì chữ số đó rơi vào hàng cao hơn, làm tổng lớn hơn.

Tính tay `4325`: chữ số `2 3 4 5`. Hai số 2 chữ số, có hai hàng chục và hai hàng đơn vị. Chữ số nhỏ nhất `2, 3` vào hàng chục; `4, 5` vào hàng đơn vị: `24 + 35` hoặc `25 + 34` hoặc `23 + 45`... đều có hàng chục `{2, 3}` và hàng đơn vị `{4, 5}`:

```
2 4 + 3 5 = 20 + 4 + 30 + 5 = 59
2 5 + 3 4 = 20 + 5 + 30 + 4 = 59
```

Tổng chỉ phụ thuộc **chữ số nào vào hàng nào**, không phụ thuộc hai chữ số hàng chục ghép với số nào ở hàng đơn vị.

In [7]:
tens = [2, 3]
ones = [4, 5]
print("tổng =", 10 * sum(tens) + sum(ones))        # 10*(2+3) + (4+5) = 59
tens2 = [4, 5]
ones2 = [2, 3]
print("đổi chỗ (chữ số lớn ở hàng chục) =", 10 * sum(tens2) + sum(ones2), "(lớn hơn)")

tổng = 59
đổi chỗ (chữ số lớn ở hàng chục) = 95 (lớn hơn)


## 5. Cách 1 — Brute Force / baseline

Thử **mọi hoán vị** của các chữ số, rồi thử **mọi điểm cắt** để tách thành hai số, lấy tổng nhỏ nhất. Số hoán vị là `d!` với `d` là số chữ số — chỉ dùng được cho `num` ít chữ số, dùng làm "máy kiểm tra" cho các cách sau.

In [8]:
from itertools import permutations

def split_bruteforce(num: int) -> int:
    s = str(num)
    best = None
    for perm in permutations(s):
        text = "".join(perm)
        for cut in range(1, len(text)):          # cắt giữa: cả hai phần đều không rỗng
            total = int(text[:cut]) + int(text[cut:])
            if best is None or total < best:
                best = total
    return best

In [9]:
print(split_bruteforce(4325))     # 59
print(split_bruteforce(687))      # 75
print(split_bruteforce(10))       # 1   (0 + 1)

59
75
1


**Độ phức tạp:** gọi `d` là số chữ số. Có `d!` hoán vị × `(d - 1)` điểm cắt × `O(d)` để ghép và đổi sang số → Time `O(d! * d^2)`; Space `O(d)`. Với `d` tới 10 (`num` lên tới `10^9`) thì `10! = 3,628,800` hoán vị — quá chậm để nộp, nhưng cho ví dụ nhỏ thì tuyệt vời để **kiểm chứng**.

**Nút thắt:** ta thử rất nhiều cách xếp trong khi tổng chỉ phụ thuộc "chữ số nào ở hàng nào" (xem mục 4). Cần một quy tắc chọn thẳng.

## 6. Ý tưởng thuật toán / bước đột phá

**Sắp xếp các chữ số tăng dần, rồi lần lượt chia luân phiên cho `num1`, `num2`:** chữ số thứ 0 → `num1`, thứ 1 → `num2`, thứ 2 → `num1`, thứ 3 → `num2`, ...

Vì sao: các chữ số nhỏ nhất được đặt ở các hàng cao nhất (đầu của `num1` và đầu của `num2`), hai chữ số nhỏ kế tiếp ở hàng kế tiếp, ... và độ dài hai số chênh nhau tối đa 1.

Quy tắc "nếu… thì…":

- Duyệt các chữ số theo thứ tự tăng dần với chỉ số `i = 0, 1, 2, ...`.
- Nếu `i` chẵn → nối chữ số vào `num1`; nếu `i` lẻ → nối vào `num2`.
- Kết quả là `num1 + num2` (đổi sang số; chữ số `0` ở đầu không sao).

In [10]:
digits = sorted(str(4325))
a = "".join(digits[0::2])
b = "".join(digits[1::2])
print(digits, "->", a, "+", b, "=", int(a) + int(b))

['2', '3', '4', '5'] -> 24 + 35 = 59


## 7. Các cách tiếp theo

Ta có thêm 3 cách: **(2)** sort + chia luân phiên bằng chuỗi, **(3)** sort + xây số trực tiếp bằng `a = a * 10 + d`, **(4)** đếm chữ số (counting sort) rồi xây số, không dùng `sorted` và không dùng chuỗi.

### Cách 2 — Sắp xếp + chia luân phiên bằng chuỗi

Giải thích từng đoạn:

1. `digits = sorted(str(num))` — list ký tự đã sắp tăng dần.
2. `digits[0::2]` và `digits[1::2]` chia luân phiên thành hai list.
3. `"".join(...)` ghép ký tự thành chuỗi rồi `int(...)` đổi sang số (chấp nhận số `0` đứng đầu).

In [11]:
def split_sort_string(num):
    digits = sorted(str(num))
    a = int("".join(digits[0::2]))
    b = int("".join(digits[1::2]))
    return a + b

print(split_sort_string(4325), split_sort_string(687))

59 75


In [12]:
class SolutionSortString:
    def splitNum(self, num: int) -> int:
        digits = sorted(str(num))
        return int("".join(digits[0::2])) + int("".join(digits[1::2]))

Độ phức tạp Cách 2: gọi `d` là số chữ số. Sort `O(d log d)`; Space `O(d)`. Với `d <= 10` coi như `O(1)`.

### Cách 3 — Sắp xếp + xây số bằng `a * 10 + d`

Thay vì ghép chuỗi, ta giữ hai biến số `a`, `b`. Mỗi chữ số `ch` (theo thứ tự tăng) được "đẩy vào cuối" một trong hai số: `a = a * 10 + ch`. Chữ số `0` ở đầu tự nhiên thành `a = 0 * 10 + 0 = 0`.

In [13]:
def split_sort_numeric(num):
    a = b = 0
    for i, ch in enumerate(sorted(str(num))):
        d = int(ch)
        if i % 2 == 0:
            a = a * 10 + d
        else:
            b = b * 10 + d
    return a + b

print(split_sort_numeric(4325), split_sort_numeric(687), split_sort_numeric(10))

59 75 1


In [14]:
class SolutionSortNumeric:
    def splitNum(self, num: int) -> int:
        a = b = 0
        for i, ch in enumerate(sorted(str(num))):
            d = int(ch)
            if i % 2 == 0:
                a = a * 10 + d
            else:
                b = b * 10 + d
        return a + b

Độ phức tạp Cách 3: Time `O(d log d)` (sort) + `O(d)`; Space `O(d)`.

### Cách 4 — Đếm chữ số (counting sort), không chuỗi

Chữ số chỉ có 10 giá trị `0..9`, nên thay `sorted` bằng mảng đếm `cnt[0..9]`: duyệt `digit` từ `0` tới `9`, mỗi chữ số lặp `cnt[digit]` lần, vẫn chia luân phiên bằng biến đếm vị trí `turn`.

In [15]:
def split_counting(num):
    cnt = [0] * 10
    while num:
        cnt[num % 10] += 1
        num //= 10
    a = b = 0
    turn = 0                               # đếm vị trí trong dãy đã sắp
    for digit in range(10):
        for _ in range(cnt[digit]):
            if turn % 2 == 0:
                a = a * 10 + digit
            else:
                b = b * 10 + digit
            turn += 1
    return a + b

print(split_counting(4325), split_counting(687), split_counting(10))

59 75 1


In [16]:
class SolutionCounting:
    def splitNum(self, num: int) -> int:
        cnt = [0] * 10
        while num:
            cnt[num % 10] += 1
            num //= 10
        a = b = 0
        turn = 0
        for digit in range(10):
            for _ in range(cnt[digit]):
                if turn % 2 == 0:
                    a = a * 10 + digit
                else:
                    b = b * 10 + digit
                turn += 1
        return a + b

Độ phức tạp Cách 4: Time `O(d + 10)` = `O(d)` (không có `log`); Space `O(1)` (mảng cố định 10 ô).

## 8. TRACE từng bước

Trace Cách 3 với `num = 687` → chữ số sắp tăng `['6', '7', '8']`.

| `i` | `ch` | vào số | `a` sau | `b` sau |
|---|---|---|---|---|
| 0 | 6 | `a` (i chẵn) | 6 | 0 |
| 1 | 7 | `b` (i lẻ) | 6 | 7 |
| 2 | 8 | `a` (i chẵn) | 68 | 7 |

Kết quả `68 + 7 = 75`. Số `a` có 2 chữ số, `b` có 1 chữ số — độ dài chênh 1 khi số chữ số lẻ.

In [17]:
def trace_split(num):
    digits = sorted(str(num))
    print("chữ số sau khi sắp:", digits)
    a = b = 0
    for i, ch in enumerate(digits):
        d = int(ch)
        if i % 2 == 0:
            a = a * 10 + d
            side = "a"
        else:
            b = b * 10 + d
            side = "b"
        print(f"i={i} ch={ch} -> vào {side}: a={a}, b={b}")
    print(f"kết quả: {a} + {b} = {a + b}")
    return a + b

trace_split(687)
trace_split(4325)

chữ số sau khi sắp: ['6', '7', '8']
i=0 ch=6 -> vào a: a=6, b=0
i=1 ch=7 -> vào b: a=6, b=7
i=2 ch=8 -> vào a: a=68, b=7
kết quả: 68 + 7 = 75
chữ số sau khi sắp: ['2', '3', '4', '5']
i=0 ch=2 -> vào a: a=2, b=0
i=1 ch=3 -> vào b: a=2, b=3
i=2 ch=4 -> vào a: a=24, b=3
i=3 ch=5 -> vào b: a=24, b=35
kết quả: 24 + 35 = 59


59

In [18]:
trace_split(1000000000)     # biên trên: chữ số 1 và chín chữ số 0

chữ số sau khi sắp: ['0', '0', '0', '0', '0', '0', '0', '0', '0', '1']
i=0 ch=0 -> vào a: a=0, b=0
i=1 ch=0 -> vào b: a=0, b=0
i=2 ch=0 -> vào a: a=0, b=0
i=3 ch=0 -> vào b: a=0, b=0
i=4 ch=0 -> vào a: a=0, b=0
i=5 ch=0 -> vào b: a=0, b=0
i=6 ch=0 -> vào a: a=0, b=0
i=7 ch=0 -> vào b: a=0, b=0
i=8 ch=0 -> vào a: a=0, b=0
i=9 ch=1 -> vào b: a=0, b=1
kết quả: 0 + 1 = 1


1

## 9. Code nộp LeetCode

Ta nộp Cách 3 (sắp xếp + xây số trực tiếp) hoặc Cách 2 đều được; dưới đây là Cách 3.

In [19]:
class Solution:
    def splitNum(self, num: int) -> int:
        a = b = 0
        for i, ch in enumerate(sorted(str(num))):
            d = int(ch)
            if i % 2 == 0:
                a = a * 10 + d
            else:
                b = b * 10 + d
        return a + b

print(Solution().splitNum(4325))    # 59
print(Solution().splitNum(687))     # 75

59
75


## 10. Vì sao thuật toán đúng

Gọi `d` là số chữ số. Tổng `num1 + num2` bằng tổng của (chữ số × giá trị hàng) trên cả hai số.

1. **Độ dài cân bằng tốt nhất:** nếu `num1` có `p` chữ số và `num2` có `q` chữ số (`p + q = d`), các "giá trị hàng" là `10^0, ..., 10^(p-1)` và `10^0, ..., 10^(q-1)`. Khi `p` và `q` càng lệch nhau, hàng cao nhất `10^(max(p,q) - 1)` càng lớn; hoán đổi để cân bằng (`|p - q| <= 1`) làm mọi hàng cao nhất nhỏ nhất có thể (do chỉ chuyển một chữ số từ hàng cao ở số dài sang số ngắn ở hàng thấp hơn). Do đó dùng `p = ceil(d / 2)`, `q = floor(d / 2)`.
2. **Gán chữ số vào hàng — bất đẳng thức hoán vị (rearrangement inequality):** sau khi chốt các giá trị hàng, tổng `sum(digit_k * weight_k)` nhỏ nhất khi chữ số **nhỏ nhất** gắn với trọng số **lớn nhất**, chữ số lớn nhất gắn với trọng số nhỏ nhất. Nếu có cặp ngược (chữ số lớn hơn ở hàng lớn hơn), **hoán đổi** hai chữ số đó không làm tổng tăng — vậy có thể sắp lại cho tới khi "nhỏ ghép lớn".
3. **Trọng số theo thứ tự giảm dần là:** `10^(p-1)` (số dài), `10^(q-1)` (số ngắn, bằng hoặc kém 10 lần), rồi lặp lại... Khi `p = q` mỗi trọng số xuất hiện hai lần liên tiếp (hai hàng chục, hai hàng đơn vị...). Khi `p = q + 1` thì hàng cao nhất của số dài một mình, sau đó mỗi trọng số hai lần. Trong cả hai trường hợp, thứ tự "luân phiên `num1, num2, num1, ...`" khớp đúng thứ tự trọng số giảm dần.
4. **Chữ số 0:** đề cho phép số có `0` ở đầu, nên không cần xử lý riêng; chữ số `0` đứng đầu dãy tăng dần như mọi chữ số khác (và `int("07") = 7`).

Vậy: sắp chữ số tăng dần và chia luân phiên cho kết quả nhỏ nhất. Ta cũng **kiểm chứng bằng vét cạn** ở mục 12.

## 11. Phân tích Time / Space Complexity

Gọi `d` là số chữ số của `num` (`d <= 10`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn hoán vị | `O(d! * d^2)` | `O(d)` |
| 2. Sort + luân phiên (chuỗi) | `O(d log d)` | `O(d)` |
| 3. Sort + `a * 10 + d` | `O(d log d)` | `O(d)` |
| 4. Counting sort | `O(d)` | `O(1)` (mảng 10 ô) |

Theo giá trị `N = num`, `d = O(log N)`, nên cách 2, 3 là `O(log N log log N)` và cách 4 là `O(log N)`. Với `N <= 10^9` mọi cách (trừ vét cạn) đều tức thì.

## 12. Bộ test

Test 1: ví dụ đề và các edge case: `10` (nhỏ nhất, có `0`), số có chữ số lặp, số toàn chữ số bằng nhau, số kết thúc bằng nhiều `0`, và biên trên `10^9`.

In [20]:
impls = [
    ("sort_string", split_sort_string),
    ("sort_numeric", split_sort_numeric),
    ("counting", split_counting),
    ("class_sort_string", SolutionSortString().splitNum),
    ("class_sort_numeric", SolutionSortNumeric().splitNum),
    ("class_counting", SolutionCounting().splitNum),
    ("submit", Solution().splitNum),
]
tests = [
    (4325, 59),
    (687, 75),
    (10, 1),             # 0 + 1
    (11, 2),             # 1 + 1
    (99, 18),            # 9 + 9
    (100, 1),            # digits 0,0,1 -> a=01=1, b=0
    (1000000000, 1),     # biên trên
    (1234, 37),          # 13 + 24
    (5555, 110),         # 55 + 55
    (9081, 27),          # sắp 0,1,8,9 -> a = 08 = 8, b = 19 -> 27
]
for name, f in impls:
    for num, expected in tests:
        assert f(num) == expected, (name, num, f(num), expected)
    print(name, "PASS", len(tests), "test")
for num, expected in tests:        # xác nhận bảng đáp án viết tay bằng vét cạn (số <= 5 chữ số)
    if num < 10 ** 5:
        assert split_bruteforce(num) == expected, (num, split_bruteforce(num), expected)
print("bảng đáp án khớp vét cạn")

sort_string PASS 10 test
sort_numeric PASS 10 test
counting PASS 10 test
class_sort_string PASS 10 test
class_sort_numeric PASS 10 test
class_counting PASS 10 test
submit PASS 10 test
bảng đáp án khớp vét cạn


Test 2: so sánh với vét cạn trên **mọi số** từ `10` đến `9999` (4 chữ số trở xuống: nhiều ca chữ số lặp và chữ số `0`), và ngẫu nhiên có seed cho số tới 7 chữ số. Mọi input đều hợp lệ (`10 <= num <= 10^9`).

In [21]:
for num in range(10, 10000):
    expected = split_bruteforce(num)
    for name, f in impls:
        assert f(num) == expected, (name, num)
print("đã so khớp vét cạn với mọi num trong [10, 9999]")

đã so khớp vét cạn với mọi num trong [10, 9999]


In [22]:
import random
random.seed(2578)
for _ in range(60):
    num = random.randint(10 ** 4, 9999999)           # 5 đến 7 chữ số
    expected = split_bruteforce(num)
    for name, f in impls:
        assert f(num) == expected, (name, num)
print("60 test ngẫu nhiên 5-7 chữ số khớp vét cạn")

60 test ngẫu nhiên 5-7 chữ số khớp vét cạn


Test 3: các số lớn (8-10 chữ số, không vét cạn được) — chỉ so các cách nhanh với nhau; thêm kiểm tra tính chất: kết quả `<= num` và hàm bất biến khi **hoán vị chữ số** của `num` (vì chỉ phụ thuộc vào tập chữ số).

In [23]:
random.seed(5)
for _ in range(2000):
    num = random.randint(10 ** 7, 10 ** 9)
    results = {f(num) for _, f in impls}
    assert len(results) == 1, (num, results)
    assert Solution().splitNum(num) <= num
    # hoán vị chữ số, vẫn giữ chữ số đầu khác 0 để hợp lệ
    chars = list(str(num))
    random.shuffle(chars)
    if chars[0] != "0":
        assert Solution().splitNum(int("".join(chars))) == Solution().splitNum(num)
print("2000 test lớn + tính chất bất biến khi hoán vị chữ số: PASS")

2000 test lớn + tính chất bất biến khi hoán vị chữ số: PASS


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên sắp xếp (tách đôi dãy chữ số theo thứ tự gốc)

In [24]:
def wrong_no_sort(num):
    s = str(num)
    return int(s[:len(s) // 2]) + int(s[len(s) // 2:])

print(wrong_no_sort(4325), "(đáp án đúng là 59)")      # 43 + 25 = 68

68 (đáp án đúng là 59)


Vì sao sai: thứ tự gốc của chữ số chẳng liên quan tới tổng nhỏ nhất. Ta được **sắp xếp lại** chữ số tự do, và phải sắp để chữ số nhỏ vào hàng cao.

### Lỗi 2: sắp xếp **giảm dần** (chữ số lớn vào hàng cao)

In [25]:
def wrong_desc(num):
    digits = sorted(str(num), reverse=True)
    return int("".join(digits[0::2])) + int("".join(digits[1::2]))

print(wrong_desc(4325), "(đáp án đúng là 59)")        # 53 + 42 = 95

95 (đáp án đúng là 59)


Vì sao sai: đặt `5, 4` ở hàng chục và `3, 2` ở hàng đơn vị cho `53 + 42 = 95` — đúng bài toán **lớn nhất**, ngược với đề (nhỏ nhất).

### Lỗi 3: chia đôi dãy đã sắp (nửa đầu cho `num1`, nửa sau cho `num2`) thay vì chia luân phiên

In [26]:
def wrong_halves(num):
    digits = sorted(str(num))
    h = len(digits) // 2
    return int("".join(digits[:h])) + int("".join(digits[h:]))

print(wrong_halves(4325), "(đáp án đúng là 59)")      # 23 + 45 = 68

68 (đáp án đúng là 59)


Vì sao sai: `23 + 45 = 68`: chữ số `2` và `3` nằm ở hàng chục và hàng đơn vị của cùng một số, nên chữ số `3` (nhỏ) rơi vào hàng đơn vị còn `4` (lớn hơn) lên hàng chục. Phải để `2` và `3` **cùng ở hàng chục** của hai số khác nhau → chia **luân phiên**.

### Lỗi 4: cộng **chuỗi** thay vì cộng **số**

In [27]:
digits = sorted(str(4325))
a = "".join(digits[0::2])
b = "".join(digits[1::2])
print(a + b, "<- đây là phép NỐI chuỗi '24' + '35' = '2435'")
print(int(a) + int(b), "<- phải đổi sang int rồi cộng")

2435 <- đây là phép NỐI chuỗi '24' + '35' = '2435'
59 <- phải đổi sang int rồi cộng


Vì sao sai: `"24" + "35"` nối thành `"2435"`; phải `int(a) + int(b) = 59`.

### Lỗi 5: dùng `set` để loại trùng chữ số (mất chữ số lặp)

In [28]:
def wrong_set(num):
    digits = sorted(set(str(num)))          # set bỏ chữ số lặp!
    return int("".join(digits[0::2])) + int("".join(digits[1::2] or ["0"]))

print(wrong_set(5555), "(đáp án đúng là", Solution().splitNum(5555), ")")   # chỉ còn một chữ số 5
print(wrong_set(4325), "(trùng khớp vì các chữ số đều khác nhau)")

5 (đáp án đúng là 110 )
59 (trùng khớp vì các chữ số đều khác nhau)


Vì sao sai: đề yêu cầu **giữ đủ số lần xuất hiện** của mỗi chữ số (hoán vị của `num`). `set` chỉ giữ mỗi chữ số một lần nên `5555` thành `5` → kết quả `5` thay vì `110`. Chỉ cần `sorted(str(num))`, không cần `set`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tách chữ số thành hai số, tổng **nhỏ nhất**" → **sắp chữ số tăng dần rồi chia luân phiên**. Nếu đề hỏi tổng **lớn nhất** → sắp giảm dần và vẫn luân phiên? Hãy tự suy luận bằng bất đẳng thức hoán vị chứ đừng học thuộc (bài tập 5 và mục 15 gợi ý).
- Từ khoá "hoán vị các chữ số", "leading zero được phép" → bài về **hàng (place value)** và **tham lam**.
- Muốn tránh chuỗi: dùng `cnt[10]` và `a = a * 10 + d`.
- Luôn kiểm chứng ý tưởng greedy bằng **vét cạn trên số nhỏ** (như mục 12) — đây là thói quen rất đáng có.
- `"".join(digits[0::2])`, `"".join(digits[1::2])` là cách chia luân phiên gọn nhất.

## 15. Biến thể của bài

1. Tách thành **`k` số** (không chỉ hai) sao cho tổng nhỏ nhất: sắp tăng dần và chia **vòng tròn** cho `k` số.
2. **Tổng lớn nhất:** các chữ số lớn vào hàng cao nhất; nhưng khi không bắt buộc cân bằng độ dài, tự phân tích kỹ (số dài hơn một chữ số làm tổng lớn hơn).
3. **Không được có 0 ở đầu** (số khác 0 phải bắt đầu bằng chữ số khác 0): bài khó hơn, phải xử lý riêng chữ số 0.
4. Tách thành hai số có **cùng số chữ số** (đề chỉ cho `num` có chữ số chẵn).
5. Tách `num` thành hai số **liên tiếp** (không hoán vị): quay lại bài toán chia mảng.
6. Biến thể trong hệ cơ số `b` khác 10: cùng ý tưởng, trọng số là `b^k`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn hoán vị | thử mọi hoán vị, mọi điểm cắt | `O(d! * d^2)` | `O(d)` | số nhỏ, làm oracle kiểm chứng |
| 2. Sort + luân phiên (chuỗi) | `digits[0::2]`, `digits[1::2]` | `O(d log d)` | `O(d)` | viết nhanh, dễ đọc nhất |
| 3. Sort + `a * 10 + d` | xây số bằng toán | `O(d log d)` | `O(d)` | không cần ghép chuỗi |
| 4. Counting sort | đếm `cnt[0..9]` rồi xây | `O(d)` | `O(1)` | muốn tối ưu tuyệt đối / không dùng sort |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().splitNum(2932)` bằng bao nhiêu?

**Bài 2 (tính tay):** tính `splitNum(5009)`: sắp các chữ số, chia luân phiên, ghi `a`, `b`, tổng.

**Bài 3 (điền TODO):** hoàn thành `alternate_split(digits)` nhận list chữ số **đã sắp tăng dần** (dạng int) và trả về cặp `(a, b)` là hai số tạo ra bằng cách chia luân phiên.

**Bài 4 (tự viết lại):** viết `split_min_sum_counting(num)` bằng mảng đếm `cnt[10]`, **không** dùng `sorted` và `str`.

**Bài 5 (mini challenge):** viết `split_k(num, k)` tách các chữ số thành đúng `k` số (mỗi số có ít nhất 1 chữ số, cho phép số 0 ở đầu) để tổng nhỏ nhất, bằng cách sắp tăng dần và chia **vòng tròn** (chữ số thứ `i` vào số thứ `i % k`).

In [29]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().splitNum(2932))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def alternate_split(digits):
    a = b = 0
    # TODO: duyệt (i, d) bằng enumerate(digits):
    #   - nếu i chẵn: a = a * 10 + d
    #   - nếu i lẻ  : b = b * 10 + d
    return a, b

52


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** luôn sắp chữ số **tăng dần**; chữ số thứ 0 và thứ 1 là hai chữ số nhỏ nhất, nằm ở hàng cao nhất của hai số.

**Mức 2 (vừa):** Bài 4: dựng `cnt` bằng `while num: cnt[num % 10] += 1; num //= 10`, rồi hai vòng `for digit in range(10)` và `for _ in range(cnt[digit])` với biến `turn`. Bài 5: giữ list `parts = [0] * k` rồi `parts[i % k] = parts[i % k] * 10 + d`.

**Mức 3 (gần đáp án):** Bài 1: sắp `2, 2, 3, 9`: `a = 23`, `b = 29` → `52`. Bài 2: sắp `0, 0, 5, 9`: `a = 05 = 5`, `b = 09 = 9` → `14`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1: sắp 2,2,3,9 -> a = 23, b = 29
assert Solution().splitNum(2932) == 52 == split_bruteforce(2932)

# Bài 2: sắp 0,0,5,9 -> a = 05 = 5, b = 09 = 9
assert Solution().splitNum(5009) == 14 == split_bruteforce(5009)

# Bài 3
def alternate_split_answer(digits):
    a = b = 0
    for i, d in enumerate(digits):
        if i % 2 == 0:
            a = a * 10 + d
        else:
            b = b * 10 + d
    return a, b

assert alternate_split_answer([2, 3, 4, 5]) == (24, 35)
assert alternate_split_answer([6, 7, 8]) == (68, 7)

# Bài 4
def split_min_sum_counting(num):
    cnt = [0] * 10
    while num:
        cnt[num % 10] += 1
        num //= 10
    a = b = 0
    turn = 0
    for digit in range(10):
        for _ in range(cnt[digit]):
            if turn % 2 == 0:
                a = a * 10 + digit
            else:
                b = b * 10 + digit
            turn += 1
    return a + b

for n in [10, 4325, 687, 5009, 1000000000, 99, 2932]:
    assert split_min_sum_counting(n) == Solution().splitNum(n)

# Bài 5
from itertools import permutations
def split_k(num, k):
    parts = [0] * k
    for i, ch in enumerate(sorted(str(num))):
        parts[i % k] = parts[i % k] * 10 + int(ch)
    return sum(parts)

assert split_k(4325, 2) == Solution().splitNum(4325) == 59

def brute_three(num):
    s = str(num)
    best = None
    for perm in permutations(s):
        t = "".join(perm)
        for i in range(1, len(t)):
            for j in range(i + 1, len(t)):
                total = int(t[:i]) + int(t[i:j]) + int(t[j:])
                if best is None or total < best:
                    best = total
    return best

for n in [123, 9876, 10203, 55555, 120345, 906]:
    assert split_k(n, 3) == brute_three(n), n
print("tất cả đáp án bài tập: PASS")

tất cả đáp án bài tập: PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "sắp tăng dần rồi chia luân phiên":**

```python
digits = sorted(str(num))                       # chữ số tăng dần
a = int("".join(digits[0::2]))                  # vị trí chẵn -> num1
b = int("".join(digits[1::2]))                  # vị trí lẻ  -> num2
return a + b
```

**Checklist trước khi submit:**

- [ ] Sắp xếp **tăng dần** (không phải giảm dần)?
- [ ] Chia **luân phiên** (`[0::2]`, `[1::2]`), không phải chia đôi dãy?
- [ ] Đã `int(...)` trước khi cộng (không cộng chuỗi)?
- [ ] Giữ **đủ số lần xuất hiện** của mọi chữ số (không dùng `set`), kể cả chữ số `0` (đề cho phép số có `0` đứng đầu)?
- [ ] Đã thử `num = 10`, số có chữ số lặp, số có chữ số lẻ độ dài (`687`), và `10^9`?
- [ ] Đã đối chiếu với vét cạn trên số nhỏ?
- [ ] Chữ ký đúng `splitNum(self, num: int) -> int`?